# Experiment: ResNet50 Baseline

**Student ID:** 25509225  
**Experiment:** Baseline - Standard ResNet50 with single FC layer (2048→10)  
**Purpose:** Establish baseline performance for comparison with customized models.

## Cell 1: Load Modules

In [ ]:
%run ./ResNet50_modules.ipynb

## Cell 2: Configuration

In [ ]:
STUDENT_ID = "25509225"

DATA_ROOT = f"/home/sagemaker-user/CNN_A2/data/{STUDENT_ID}/Image_Classification/split_dataset"

BATCH_SIZE = 16
AUGMENTATION_TYPE = 'none'  # Can be changed to 'standard' or 'enhanced'
USE_PRETRAINED = True       # Can be changed to False

output_dir = Path(f'outputs/classification_baseline')
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("EXPERIMENT: ResNet50 Baseline")
print("=" * 80)
print(f'\nConfiguration:')
print(f'  Data Root: {DATA_ROOT}')
print(f'  Batch Size: {BATCH_SIZE}')
print(f'  Augmentation: {AUGMENTATION_TYPE}')
print(f'  Pretrained: {USE_PRETRAINED}')
print(f'  Output Dir: {output_dir}')

## Cell 3: Step 1 - Load Data

In [ ]:
print("\n[1/5] Loading data...")
train_loader, val_loader, test_loader, class_names = create_classification_dataloaders(
    DATA_ROOT, 
    batch_size=BATCH_SIZE, 
    augmentation_type=AUGMENTATION_TYPE
)
print(f'Classes: {class_names}')
print(f'Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}')

aug_descriptions = {
    'none': 'No augmentation (basic preprocessing only)',
    'standard': 'Standard (Rotation 15°, ColorJitter 0.2)',
    'enhanced': 'Enhanced (Rotation 20°, ColorJitter 0.3+hue, RandomAffine)'
}
print(f'Data augmentation: {aug_descriptions[AUGMENTATION_TYPE]}')

## Cell 4: Step 2 - Initialize Model

In [ ]:
print("\n[2/5] Initializing model...")

model_config = BASELINE_CONFIG.copy()
model_config['pretrained'] = USE_PRETRAINED

if model_config['pretrained']:
    print('Architecture: Standard ResNet50 with ALL layers trainable (NO freezing)')
    print('Pretrained: YES (ImageNet weights)')
else:
    print('Architecture: Standard ResNet50 with ALL layers trainable (NO freezing)')
    print('Pretrained: NO (Training from scratch)')

model = ResNet50Classifier(**model_config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}, Trainable: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)')

# Print model summary
trainer_temp = ClassificationTrainer(model, config=TRAINING_CONFIG_BASELINE)
trainer_temp.print_model_summary()

## Cell 5: Step 3 - Train

In [ ]:
print("\n[3/5] Training...")
trainer = trainer_temp  # Reuse trainer from Cell 4
criterion = torch.nn.CrossEntropyLoss(label_smoothing=TRAINING_CONFIG_BASELINE.label_smoothing)

history = trainer.train(
    train_loader, 
    val_loader, 
    criterion,
    str(output_dir)  # Only saves best_model.pth
)
print(f'Best Val Acc: {trainer.best_val_acc:.4f}')

## Cell 6: Load Best Model

In [ ]:
print("\nLoading best model for evaluation...")
best_model_path = output_dir / 'best_model.pth'
if best_model_path.exists():
    checkpoint = torch.load(best_model_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(trainer.device)
    print(f'✓ Loaded best model from epoch {checkpoint["epoch"]} (Val Acc: {checkpoint["val_acc"]:.4f})')
else:
    print('Warning: Best model checkpoint not found, using current model')

## Cell 7: Step 4 - Evaluate

In [ ]:
print("\n[4/5] Evaluating on test set...")
evaluator = ClassificationEvaluator(class_names)
metrics = evaluator.evaluate(model, test_loader)  # No output_dir parameter

print("\n=== Test Set Metrics ===")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Precision (weighted): {metrics['precision_weighted']:.4f}")
print(f"Recall (weighted): {metrics['recall_weighted']:.4f}")
print(f"F1 (weighted): {metrics['f1_weighted']:.4f}")
print(f"\nPrecision (macro): {metrics['precision_macro']:.4f}")
print(f"Recall (macro): {metrics['recall_macro']:.4f}")
print(f"F1 (macro): {metrics['f1_macro']:.4f}")

## Cell 8: Display Confusion Matrix

In [ ]:
print("\n=== Confusion Matrix ===")
fig, ax = plt.subplots(figsize=(10, 8))
cm_array = metrics['confusion_matrix']
sns.heatmap(cm_array, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Cell 9: Step 5 - Display Training Curves

In [ ]:
print("\n[5/5] Training History Analysis")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['history']['epoch'], history['history']['train_loss'], 
             label='Train Loss', linewidth=2)
axes[0].plot(history['history']['epoch'], history['history']['val_loss'], 
             label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['history']['epoch'], history['history']['train_acc'], 
             label='Train Acc', linewidth=2)
axes[1].plot(history['history']['epoch'], history['history']['val_acc'], 
             label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training & Validation Accuracy', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 10: Overfitting Analysis

In [ ]:
print("\n=== Overfitting Analysis ===")
analysis = evaluator.analyze_overfitting(history['history'])
print(f"Pattern: {analysis['pattern']}")
print(f"Train-Val Accuracy Gap: {analysis['gap']:.4f}")
print(f"Recommendation: {analysis['recommendation']}")

## Cell 11: Final Summary

In [ ]:
print("\n" + "=" * 80)
print("EXPERIMENT COMPLETED")
print("=" * 80)
print(f"\nExperiment: ResNet50 Baseline")
print(f"Architecture: Standard ResNet50 with single FC layer (2048→10)")
print(f"Pretrained: {model_config['pretrained']}")
print(f"Data Augmentation: {AUGMENTATION_TYPE}")
print(f"Best Val Accuracy: {trainer.best_val_acc:.4f}")
print(f"Test Accuracy: {metrics['accuracy']:.4f}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"\nResults saved to: {output_dir}")
print(f"  - best_model.pth ✓")